# 🔒 Anonymized Edge Surveillance
**INE2 SmartICT — SIPM Project 2**

| Cellule | Rôle |
|---------|------|
| **1** | Vérification dépendances |
| **2** | Test webcam + benchmark FPS |
| **3** | État de la Whitelist (Photos storage) |
| **4** | ⭐ **Enrôlement Interactif** (Nom + 30 captures) |
| **5** | 🧠 **Entraînement Global** (Génération des embeddings) |
| **6** | Inspection du cache d'embeddings |
| **7** | Benchmark pipeline complet |
| **8** | 🚀 Lancement serveur WebRTC |

> ⚠️ **Exécute les cellules dans l'ordre la première fois.**

---
## Cellule 1 — Vérification de l'environnement

In [1]:
import sys, importlib
from pathlib import Path

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'📁 Racine projet : {ROOT}')
print(f'🐍 Python        : {sys.version.split()[0]}\n')

REQUIRED = {
    'cv2'       : 'opencv-python',
    'mediapipe' : 'mediapipe',
    'numpy'     : 'numpy',
    'aiortc'    : 'aiortc',
    'aiohttp'   : 'aiohttp',
    'deepface'  : 'deepface',
}

all_ok = True
for module, pkg in REQUIRED.items():
    try:
        m   = importlib.import_module(module)
        ver = getattr(m, '__version__', '?')
        print(f'  ✅ {pkg:<22} v{ver}')
    except ImportError:
        print(f'  ❌ {pkg:<22} → pip install {pkg}')
        all_ok = False

print()
if all_ok:
    print('✅ Toutes les dépendances sont disponibles.')
else:
    print('❌ Installe les packages manquants puis relance cette cellule.')

📁 Racine projet : C:\Users\CLOUD TECHNOLOGY\Downloads\Cours_inpt\Streaming\code\Anonymized-Edge-Surveillance
🐍 Python        : 3.10.11

  ✅ opencv-python          v4.13.0
  ✅ mediapipe              v0.10.33
  ✅ numpy                  v2.2.6
  ✅ aiortc                 v1.14.0
  ✅ aiohttp                v3.13.4
  ✅ deepface               v0.0.99

✅ Toutes les dépendances sont disponibles.


---
## Cellule 2 — Vérification webcam

In [2]:
import cv2
import time
import numpy as np

# 1. Ouverture webcam
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print('❌ Erreur : Impossible d\'ouvrir la webcam.')
else:
    print('✅ Webcam OK : 640x480 pixel(s)')
    
    # 2. Benchmark (test sur 30 images)
    durations = []
    print('⏱  Calcul de la performance sur 30 frames...')
    
    for _ in range(30):
        t0 = time.perf_counter()
        ret, frame = cap.read()
        if not ret: break
        
        # Test simulation d'un traitement simple (ex: conversion gris)
        _ = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        t1 = time.perf_counter()
        durations.append((t1 - t0) * 1000) # Stockage en millisecondes
    
    cap.release()
    
    # 3. Calcul Resultat
    avg_ms = np.mean(durations)
    fps = 1000 / avg_ms
    limit_ms = 40.0 # Objectif de latence
    
    # 4. Affichage du Verdict
    print("-" * 45)
    print(f"🔹 Temps moyen par image : {avg_ms:.2f} ms")
    print(f"🔹 Images par seconde     : {fps:.2f} FPS")
    print("-" * 45)
    
    if avg_ms <= limit_ms:
        print(f"✅ SUCCÈS : {avg_ms:.2f} ms <= {limit_ms} ms")
        print("🚀 Le pipeline est assez rapide pour le streaming fluide.")
    else:
        print(f"⚠️  ALERTE : {avg_ms:.2f} ms > {limit_ms} ms")
        print("🔴 Attention : Risque de lag dans le navigateur !")


✅ Webcam OK : 640x480 pixel(s)
⏱  Calcul de la performance sur 30 frames...
---------------------------------------------
🔹 Temps moyen par image : 51.39 ms
🔹 Images par seconde     : 19.46 FPS
---------------------------------------------
⚠️  ALERTE : 51.39 ms > 40.0 ms
🔴 Attention : Risque de lag dans le navigateur !


---
## Cellule 3 — État de la Whitelist (Dossiers photos)

In [3]:
import os
from pathlib import Path
from config.settings import WHITELIST_DIR

WHITELIST_DIR.mkdir(exist_ok=True)
subdirs = [d for d in WHITELIST_DIR.iterdir() if d.is_dir()]

print(f'📂 Dossier Whitelist : {WHITELIST_DIR}')
if not subdirs:
    print('⚠️ Aucun dossier de personne trouvé. Utilisez la Cellule 4 pour commencer.')
else:
    print(f'👥 {len(subdirs)} personne(s) dans la base :')
    for d in subdirs:
        n_photos = len(list((d / "samples").glob("*.jpg"))) if (d / "samples").exists() else 0
        print(f'  - {d.name} ({n_photos} photos)')

📂 Dossier Whitelist : C:\Users\CLOUD TECHNOLOGY\Downloads\Cours_inpt\Streaming\code\Anonymized-Edge-Surveillance\whitelist
👥 2 personne(s) dans la base :
  - marin (30 photos)
  - mustapha (30 photos)


---
## Cellule 4 — ⭐ Enrôlement Interactif (Capture de photos)

Exécutez cette cellule pour ajouter une nouvelle personne.
1. Entrez le nom dans le champ qui apparaîtra.
2. La webcam s'ouvre, appuyez sur **ESPACE** pour lancer la capture auto de 30 images.

In [3]:
import os
from pathlib import Path

# On s'assure d'être au bon endroit
ROOT = Path('.').resolve()
if (ROOT / "tools" / "enroll_person.py").exists():
    %run tools/enroll_person.py
else:
    print(f"❌ Script introuvable dans {ROOT}/tools/")



   ENRÔLEMENT NOUVELLE PERSONNE


📸 Prêt pour l'enrôlement de : MUSTAPHA
➡️  Placez-vous face à la caméra.
   Appuyez sur [ESPACE] pour lancer la capture auto de 30 images.
   Appuyez sur [Q] pour annuler.

🔄 Capture en cours dans : c:\Users\CLOUD TECHNOLOGY\Downloads\Cours_inpt\Streaming\code\Anonymized-Edge-Surveillance\whitelist\mustapha\samples ...
  Capture 30/30

✅ Enrôlement terminé !
📁 30 images sauvegardées dans : c:\Users\CLOUD TECHNOLOGY\Downloads\Cours_inpt\Streaming\code\Anonymized-Edge-Surveillance\whitelist\mustapha\samples
💡 Note : Les embeddings seront calculés lors de l'entraînement global.


---
## Cellule 5 — 🧠 Entraînement Global (Construction des Embeddings)

Cette cellule parcourt tous les dossiers de `whitelist/` et génère un embedding moyen pour chaque personne.
C'est cette étape qui construit la "mémoire" du système.

In [7]:
import pickle, numpy as np
from pathlib import Path
from deepface import DeepFace
from config.settings import WHITELIST_DIR, EMBEDDINGS_CACHE

print("🚀 Démarrage de l'entraînement global...\n")
entries = []

for person_dir in WHITELIST_DIR.iterdir():
    if not person_dir.is_dir(): continue
    
    samples_dir = person_dir / "samples"
    if not samples_dir.exists(): continue
    
    name = person_dir.name
    photos = list(samples_dir.glob("*.jpg"))
    
    if not photos:
        print(f"  ⚠️ Aucun échantillon pour {name}, ignoré.")
        continue
        
    print(f"  🧠 Traitement de {name} ({len(photos)} photos)... ")
    person_embeddings = []
    
    for photo in photos:
        try:
            rep = DeepFace.represent(
                img_path = str(photo), 
                model_name = "Facenet", 
                detector_backend = "skip",
                enforce_detection = False
            )
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            person_embeddings.append(emb)
        except:
            continue
            
    if person_embeddings:
        mean_emb = np.mean(person_embeddings, axis=0)
        mean_emb = mean_emb / (np.linalg.norm(mean_emb) + 1e-10)
        entries.append({"name": name, "embedding": mean_emb})
        print(f"    ✅ OK")
    else:
        print(f"    ❌ Erreur d'encodage")

# Sauvegarde du cache
EMBEDDINGS_CACHE.parent.mkdir(parents=True, exist_ok=True)
with open(EMBEDDINGS_CACHE, "wb") as f:
    pickle.dump(entries, f)

print(f"\n✅ Entraînement terminé ! {len(entries)} personnes encodées dans {EMBEDDINGS_CACHE}")

🚀 Démarrage de l'entraînement global...

  🧠 Traitement de marin (30 photos)... 
    ✅ OK
  🧠 Traitement de mustapha (30 photos)... 
    ✅ OK

✅ Entraînement terminé ! 2 personnes encodées dans C:\Users\CLOUD TECHNOLOGY\Downloads\Cours_inpt\Streaming\code\Anonymized-Edge-Surveillance\whitelist\.embeddings_cache.pkl


---
## Cellule 6 — Inspection du cache d'embeddings

In [5]:
import pickle, numpy as np
from config.settings import EMBEDDINGS_CACHE

if not EMBEDDINGS_CACHE.exists():
    print('❌ Cache absent — lancez la cellule 5.')
else:
    with open(EMBEDDINGS_CACHE, 'rb') as f:
        entries = pickle.load(f)
    
    print(f'✅ Cache chargé : {len(entries)} personne(s)')
    for e in entries:
        print(f"  👤 {e['name']:<15} (Vecteur dim: {len(e['embedding'])})")

✅ Cache chargé : 2 personne(s)
  👤 marin           (Vecteur dim: 128)
  👤 mustapha        (Vecteur dim: 128)


In [9]:
# Rebuild cache embeddings en 512-D (PyTorch FaceNet)
from detection.face_recognizer import get_db, TORCH_AVAILABLE

if not TORCH_AVAILABLE:
    raise RuntimeError("Installe d'abord les dépendances: py -3 -m pip install -r requirements.txt")

db = get_db()
n = db.build_from_whitelist()
if n <= 0:
    raise RuntimeError("Échec build_from_whitelist() : vérifie whitelist/*/samples")
db.save()

print(f"Cache reconstruit ✅ | personnes encodées: {n} | noms: {db.names()}")

Cache reconstruit ✅ | personnes encodées: 2 | noms: ['marin', 'mustapha']


---
## Cellule 7 — Benchmark pipeline complet

In [10]:
import cv2, time, numpy as np
from detection.face_recognizer import get_db, get_recognizer
from anonymization.privacy_modes import process_frame

db = get_db()
if not db.load():
    print('⚠️ Cache absent — lancez la cellule 5.')
else:
    print('⏳ Warmup FaceNet...')
    get_recognizer()
    print('✅ Prêt.')

⏳ Warmup FaceNet...
✅ Prêt.


---
## Cellule 8 — 🚀 Lancement du serveur WebRTC

In [1]:
import asyncio
from aiohttp import web
from streaming.webrtc_server import create_app
from detection.face_recognizer import get_db, get_recognizer

# 1. Chargement et Warmup (comme avant)
get_db().load()
get_recognizer() 

# 2. Configuration du serveur
app = create_app()
runner = web.AppRunner(app)

async def run_server():
    await runner.setup()
    site = web.TCPSite(runner, '0.0.0.0', 8080)
    print('🚀 Serveur WebRTC démarré sur http://localhost:8080')
    await site.start()
    
    # Garde la cellule active pour que le serveur tourne
    try:
        while True:
            await asyncio.sleep(3600) # Rafraîchit toutes les heures
    except asyncio.CancelledError:
        print("\n🛑 Arrêt du serveur...")
        await runner.cleanup()

# 3. Lancement asynchrone (compatible Jupyter)
loop = asyncio.get_event_loop()
server_task = loop.create_task(run_server())


🚀 Serveur WebRTC démarré sur http://localhost:8080
